# Cuadernillo grafico: datos abiertos sobre IA en instituciones publicas

Este notebook procesa `Data_2/para-datos-abiertos-datos_corregidos.csv`, una encuesta institucional sobre capacidades, adopcion, gobernanza, contratacion, capacitacion y resguardos para proyectos de inteligencia artificial en el sector publico.

El archivo contiene 121 respuestas y no incluye ponderadores muestrales, por lo que los resultados son descriptivos. En preguntas simples se usan porcentajes sobre respuestas validas; en preguntas de seleccion multiple se grafica el porcentaje del total de instituciones que marco cada alternativa.

In [ ]:

from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

DATA_PATH = Path("Data_2/para-datos-abiertos-datos_corregidos.csv")
OUTPUT_DIR = Path("Data_2/graficos_datos_abiertos_ia")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
N = len(raw)
cols = raw.columns.tolist()

# Alias cortos por posicion. Esto evita depender de encabezados largos con tildes.
C = {
    "codigo": cols[0],
    "respondent_id": cols[1],
    "cargo": cols[2],
    "genero": cols[3],
    "institucion": cols[4],
    "dependencia_ti": cols[5],
    "region": cols[6],
    "conocimiento_pnia": cols[7],
    "area_gob_datos": cols[8],
    "protocolos_datos": cols[9],
    "avance_ia": cols[10],
    "conoce_oficio_711": cols[33],
    "conoce_guia_etica": cols[36],
    "contrato_servicios": cols[53],
    "conoce_directiva": cols[72],
    "uso_directiva": cols[73],
    "perfiles_ia": cols[74],
    "necesita_capacitacion": cols[75],
    "formacion_etica": cols[83],
    "n_proyectos": cols[84],
    "expectativas": cols[86],
    "conoce_permitido_innovar": cols[97],
    "uso_permitido_innovar": cols[98],
    "uso_guia_etica": cols[99],
    "directrices_eticas": cols[101],
    "uso_cuaderno_transparencia": cols[102],
    "grado_transparencia": cols[104],
    "mitiga_sesgos": cols[105],
    "responsable_ia": cols[106],
    "conoce_algoritmos_publicos": cols[107],
    "subio_algoritmos": cols[108],
}

groups = {
    "motivos_no_uso": cols[12:23],
    "barreras_uso": cols[24:33],
    "ambitos_proyectos": cols[40:53],
    "barreras_contratacion": cols[55:72],
    "temas_capacitacion": cols[77:83],
    "usos_institucion": cols[88:91],
}

# Etiquetas breves y limpias para columnas de seleccion multiple.
short_names = {
    cols[12]: "Falta de conocimiento",
    cols[13]: "Costos elevados",
    cols[14]: "Falta de personal capacitado",
    cols[15]: "Falta de infraestructura tecnologica",
    cols[16]: "Ciberseguridad",
    cols[17]: "Resistencia al cambio",
    cols[18]: "Incertidumbre legal y etica",
    cols[19]: "IA percibida como poco relevante",
    cols[20]: "Problemas eticos",
    cols[21]: "Falta de apoyo estrategico",
    cols[22]: "Otro",
    cols[24]: "Falta de personal capacitado",
    cols[25]: "Infraestructura tecnologica limitada",
    cols[26]: "Integracion con sistemas existentes",
    cols[27]: "Presupuesto insuficiente",
    cols[28]: "Datos de calidad insuficientes",
    cols[29]: "Ciberseguridad y fuga de datos",
    cols[30]: "Escalabilidad",
    cols[31]: "Gestion del cambio tecnologico",
    cols[32]: "Otro",
    cols[40]: "Automatizacion y gestion operativa",
    cols[41]: "Interaccion con ciudadania",
    cols[42]: "Analitica para decisiones",
    cols[43]: "Fiscalizacion",
    cols[44]: "Fraude, anomalias y ciberseguridad",
    cols[45]: "Focalizacion social",
    cols[46]: "Planificacion urbana e infraestructura",
    cols[47]: "Justicia y seguridad publica",
    cols[48]: "Politicas publicas basadas en datos",
    cols[49]: "Talento humano y capacitacion",
    cols[50]: "Emergencias y desastres",
    cols[51]: "Monitoreo y evaluacion",
    cols[52]: "Otro",
    cols[55]: "Marco legal poco claro",
    cols[56]: "Burocracia excesiva",
    cols[57]: "Capacidad de adquisicion limitada",
    cols[58]: "Sin estandares de contratacion",
    cols[59]: "Infraestructura tecnologica",
    cols[60]: "Datos de calidad escasos",
    cols[61]: "Integracion de sistemas",
    cols[62]: "Falta de personal capacitado",
    cols[63]: "Desconocimiento tecnologico",
    cols[64]: "Resistencia cultural",
    cols[65]: "Restricciones presupuestarias",
    cols[66]: "Costo elevado de soluciones",
    cols[67]: "Privacidad y etica",
    cols[68]: "Confianza en proveedores",
    cols[69]: "Vision estrategica limitada",
    cols[70]: "Sin metricas claras",
    cols[71]: "Otro",
    cols[77]: "Fundamentos, etica y gobernanza",
    cols[78]: "Habilidades tecnicas",
    cols[79]: "Gestion y evaluacion de proyectos",
    cols[80]: "Aplicaciones practicas",
    cols[81]: "Cultura y colaboracion",
    cols[82]: "Otro",
    cols[88]: "Procesos internos",
    cols[89]: "Diseno y entrega de servicios publicos",
    cols[90]: "Apoyo a politicas publicas",
}


def strip_accents(text):
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", str(text))
        if not unicodedata.combining(ch)
    )


def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return np.nan
    normalized = strip_accents(text).lower()
    if normalized == "si":
        return "Si"
    return text


def valid_series(col):
    return raw[col].map(clean_text)


def label_for(col):
    return short_names.get(col, col.replace("Otro (especifique)", "Otro"))


def one_way(col, include_missing=False, order=None):
    s = valid_series(col)
    if include_missing:
        s = s.fillna("Sin respuesta")
    else:
        s = s.dropna()
    counts = s.value_counts()
    if order:
        counts = counts.reindex([x for x in order if x in counts.index]).dropna()
    tab = counts.rename_axis("respuesta").reset_index(name="n")
    base = int(tab["n"].sum()) if len(tab) else 0
    tab["porcentaje"] = tab["n"] / base * 100 if base else np.nan
    return tab


def multi_select(columns, denominator=N):
    rows = []
    for col in columns:
        marked = valid_series(col).notna().sum()
        rows.append({"item": label_for(col), "n": int(marked), "porcentaje": marked / denominator * 100})
    return pd.DataFrame(rows).sort_values("n", ascending=True)


def yes_share(col):
    s = valid_series(col).dropna()
    if len(s) == 0:
        return np.nan, 0
    return (s.eq("Si").mean() * 100), len(s)


def add_bar_labels(ax, fmt="{:.0f}%"):
    for patch in ax.patches:
        width = patch.get_width()
        if np.isfinite(width):
            ax.text(width + 0.8, patch.get_y() + patch.get_height()/2, fmt.format(width), va="center", ha="left", fontsize=9)


def barh_pct(tab, x="porcentaje", y="respuesta", title="", color="#3f6f73", xmax=100, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(tab[y], tab[x], color=color)
    ax.set_title(title, pad=10)
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    ax.tick_params(axis="y", labelsize=9)
    add_bar_labels(ax)
    return ax


def parse_project_count(value):
    text = clean_text(value)
    if text is np.nan or pd.isna(text):
        return np.nan
    low = text.lower()
    low_ascii = strip_accents(low)
    words = {"ningun": 0, "uno": 1, "una": 1, "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5, "seis": 6}
    for word, num in words.items():
        if word in low_ascii:
            return num
    nums = [int(x) for x in re.findall(r"\d+", low)]
    if not nums:
        return np.nan
    if " y " in low and len(nums) >= 2:
        return sum(nums[:2])
    return nums[0]

print(f"Filas: {raw.shape[0]:,}")
print(f"Columnas: {raw.shape[1]:,}")
raw[[C["institucion"], C["region"], C["conocimiento_pnia"], C["avance_ia"]]].head()


## Graficos generados

Las celdas siguientes generan los graficos y los guardan como PNG en `Data_2/graficos_datos_abiertos_ia/`.

In [ ]:

# 1. Composicion de la muestra institucional
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
region = one_way(C["region"], include_missing=True)
region_top = pd.concat([
    region.head(7),
    pd.DataFrame([{"respuesta": "Otras regiones", "n": region.iloc[7:]["n"].sum(), "porcentaje": region.iloc[7:]["n"].sum()/N*100}])
]).sort_values("porcentaje")
barh_pct(region_top, title="Distribucion regional de respuestas", color="#4c7899", ax=axes[0])
dep = one_way(C["dependencia_ti"]).sort_values("porcentaje")
barh_pct(dep, title="Dependencia de la Unidad TI", color="#6f8f4e", ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_composicion_muestra.png", bbox_inches="tight")
plt.show()

# 2. Conocimiento de politica nacional y avance institucional en IA
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
knowledge_order = ["Bajo", "Medio", "Alto"]
knowledge = one_way(C["conocimiento_pnia"], order=knowledge_order).sort_values("porcentaje")
avance = one_way(C["avance_ia"]).sort_values("porcentaje")
barh_pct(knowledge, title="Conocimiento de la Politica Nacional de IA", color="#b95f53", ax=axes[0])
barh_pct(avance, title="Grado de avance en el uso de IA", color="#3f6f73", ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_conocimiento_y_avance_ia.png", bbox_inches="tight")
plt.show()

# 3. Gobernanza de datos
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
gob = one_way(C["area_gob_datos"]).sort_values("porcentaje")
prot = one_way(C["protocolos_datos"]).sort_values("porcentaje")
barh_pct(gob, title="Area o persona dedicada a gobierno de datos", color="#4c7899", ax=axes[0])
barh_pct(prot, title="Protocolos de gobernanza e intercambio de datos\n(base: respuestas validas)", color="#8a6fb0", ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_gobernanza_datos.png", bbox_inches="tight")
plt.show()

# 4. Motivos para no usar IA y barreras generales de uso
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
no_uso = multi_select(groups["motivos_no_uso"])
barreras = multi_select(groups["barreras_uso"])
barh_pct(no_uso.tail(10), y="item", title="Motivos declarados para no usar IA\n(% del total de instituciones)", color="#b95f53", xmax=max(45, no_uso["porcentaje"].max()+8), ax=axes[0])
barh_pct(barreras.tail(9), y="item", title="Principales barreras para usar servicios de IA\n(% del total de instituciones)", color="#c2863a", xmax=max(45, barreras["porcentaje"].max()+8), ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_barreras_adopcion_ia.png", bbox_inches="tight")
plt.show()

# 5. Ambitos de proyectos donde se ha usado IA
ambitos = multi_select(groups["ambitos_proyectos"])
fig, ax = plt.subplots(figsize=(10.5, 6.3))
barh_pct(ambitos.tail(12), y="item", title="Ambitos o tipos de proyectos con uso de IA\n(% del total de instituciones)", color="#3f6f73", xmax=max(40, ambitos["porcentaje"].max()+8), ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_ambitos_proyectos_ia.png", bbox_inches="tight")
plt.show()

# 6. Contratacion de servicios IA y barreras de contratacion
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))
contrato = one_way(C["contrato_servicios"]).sort_values("porcentaje")
barh_pct(contrato, title="Contratacion de servicios para proyectos IA\n(base: respuestas validas)", color="#4c7899", ax=axes[0])
barriers_contract = multi_select(groups["barreras_contratacion"])
barh_pct(barriers_contract.tail(10), y="item", title="Barreras administrativas y tecnicas de contratacion\n(% del total de instituciones)", color="#b95f53", xmax=max(30, barriers_contract["porcentaje"].max()+8), ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_contratacion_y_barreras.png", bbox_inches="tight")
plt.show()

# 7. Proyectos, expectativas y usos institucionales
fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.2))
proj_num = valid_series(C["n_proyectos"]).map(parse_project_count).dropna()
proj_bucket = pd.cut(proj_num, bins=[-0.1, 0, 1, 2, 4, 100], labels=["0", "1", "2", "3-4", "5+"])
proj_tab = proj_bucket.value_counts().sort_index().rename_axis("respuesta").reset_index(name="n")
proj_tab["porcentaje"] = proj_tab["n"] / proj_tab["n"].sum() * 100
barh_pct(proj_tab.sort_values("porcentaje"), title="Numero de proyectos IA\n(base: respuestas interpretables)", color="#4c7899", ax=axes[0])
exp = one_way(C["expectativas"]).sort_values("porcentaje")
barh_pct(exp, title="Cumplimiento de expectativas\n(base: respuestas validas)", color="#3f6f73", ax=axes[1])
usos = multi_select(groups["usos_institucion"])
barh_pct(usos.sort_values("porcentaje"), y="item", title="Uso declarado de IA en la institucion\n(% del total)", color="#c2863a", xmax=max(30, usos["porcentaje"].max()+8), ax=axes[2])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_proyectos_expectativas_usos.png", bbox_inches="tight")
plt.show()



## Vista rapida de los PNG

![Composicion de la muestra](Data_2/graficos_datos_abiertos_ia/01_composicion_muestra.png)

![Conocimiento y avance IA](Data_2/graficos_datos_abiertos_ia/02_conocimiento_y_avance_ia.png)

![Gobernanza de datos](Data_2/graficos_datos_abiertos_ia/03_gobernanza_datos.png)

![Barreras de adopcion](Data_2/graficos_datos_abiertos_ia/04_barreras_adopcion_ia.png)

![Ambitos de proyectos](Data_2/graficos_datos_abiertos_ia/05_ambitos_proyectos_ia.png)

![Contratacion y barreras](Data_2/graficos_datos_abiertos_ia/06_contratacion_y_barreras.png)

![Proyectos, expectativas y usos](Data_2/graficos_datos_abiertos_ia/07_proyectos_expectativas_usos.png)

## Tablas de control

Estas tablas resumen las distribuciones usadas en los graficos y sirven para revisar rapidamente los resultados.

In [ ]:

# Tablas de control principales
for titulo, col in [
    ("Conocimiento PNIA", C["conocimiento_pnia"]),
    ("Avance en IA", C["avance_ia"]),
    ("Area/persona de gobierno de datos", C["area_gob_datos"]),
    ("Contrato servicios IA", C["contrato_servicios"]),
    ("Perfiles IA en equipo", C["perfiles_ia"]),
]:
    print("\n" + titulo)
    display(one_way(col).round({"porcentaje": 1}))

print("\nTop barreras de adopcion")
display(multi_select(groups["barreras_uso"]).sort_values("n", ascending=False).head(10).round({"porcentaje": 1}))

print("\nTop ambitos de proyectos IA")
display(multi_select(groups["ambitos_proyectos"]).sort_values("n", ascending=False).head(10).round({"porcentaje": 1}))
